# Predicting Student Mental Health Score from Social Media Use and Lifestyle Habits
**Course:** ES 335: Machine Learning (Undergraduate Track)  
**Student Name:** Aryan Mangal  
**Roll Number:** 25110058  
**Project Repository:** `mental-health-score-predictor`  

---

## 1. Project Introduction

In recent years, the ubiquity of smartphones and social media platforms has transformed student routines worldwide. While digital connectivity offers educational resources and social networking, excessive screen time, frequent notifications, and late-night phone use have raised urgent concerns regarding adolescent and collegiate wellbeing.

This project investigates the relationship between digital activity (daily social media hours, phone unlock frequency, primary platform, purpose of usage), non-digital lifestyle factors (study hours, physical exercise, sleep duration), student demographics (age, gender, academic level, country), and continuous psychological wellbeing represented by the **Mental Health Score** (on a 1–10 scale).

The goal is to develop a scientifically rigorous machine learning pipeline to:
1. Evaluate whether continuous mental health scores can be reliably predicted from observable lifestyle and digital usage habits.
2. Establish transparent baselines and benchmark diverse regression algorithms (Linear, Regularized, Distance-based, and Ensemble Tree methods).
3. Test out-of-distribution generalizability across 111 distinct student nationalities.
4. Provide actionable, evidence-based daily routine recommendations constrained strictly by a 24-hour daily time budget.


## 2. Dataset Source and Citation

### Citation:
> Shiva Singh. *Student Social Media & Mental Health Impact*. Kaggle (2024).  
> URL: [https://www.kaggle.com/datasets/shivasingh4945/student-social-media-and-mental-health-impact](https://www.kaggle.com/datasets/shivasingh4945/student-social-media-and-mental-health-impact)  
> Author: Shiva Singh (Kaggle: `@shivasingh4945`)  
> License: Open Database License (ODbL) / Public Domain for Academic Research  

### Academic Attribution Statement:
We explicitly acknowledge that the survey data used in this study was collected and made publicly available by Shiva Singh on Kaggle. The author of this project did not independently collect this dataset; all analyses, preprocessing, model development, validation checks, and visualizations presented herein are our independent implementation.


## 3. Problem Statement & Scope

### Primary Research Question:
> *Given a student's self-reported social media usage, phone unlocks, primary platform, purpose of use, and core lifestyle habits (study hours, physical exercise, and sleep duration), can a machine learning regression model reliably predict their Mental Health Score, including for students from countries unseen during training?*

### Secondary Research Questions:
1. **Feature Attribution:** Which daily habits serve as the strongest predictors of student mental wellbeing?
2. **Marginal Information of Screen Time:** Does digital media usage add significant predictive power once fundamental biological and academic habits (sleep, study, and exercise) are controlled for?
3. **Target Leakage Prevention:** Does the inclusion of subjective state indicators like `Stress_Level` artificially inflate model accuracy, and how does ethical leakage-free modeling compare?

### Scope and Limitations of the Intended Claim:
- **Prediction vs. Causation:** All statistical models developed in this study estimate predictive associations, not causal relationships. For example, reduced sleep duration may be a symptom of underlying distress rather than solely its cause.
- **Self-Reported Survey Nature:** Survey responses reflect subjective recollections, which may introduce recall bias and synthetic regularization.
- **Non-Clinical Assessment:** Predictions are intended for academic study and educational self-reflection, not clinical diagnosis or medical triage.


## 4. Environment Setup and Data Ingestion


In [ ]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting parameters
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


### Loading the Raw Dataset


In [ ]:
# Path to the primary dataset
data_path = "../data/dataset.csv"
if not os.path.exists(data_path):
    data_path = "data/dataset.csv"

raw_df = pd.read_csv(data_path)
print(f"Raw Dataset Shape: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
raw_df.head()


## 5. Dataset Understanding & Exploratory Data Analysis (EDA)

We inspect data types, missing entries, duplicates, summary statistics, and categorical cardinality.


In [ ]:
# Dataset schema and completeness
print("=== Missing Value Audit ===")
print(raw_df.isnull().sum())

print("\n=== Duplicate Rows Audit ===")
duplicates_count = raw_df.duplicated().sum()
print(f"Exact Duplicate Rows Found: {duplicates_count}")

print("\n=== Column Data Types ===")
print(raw_df.dtypes)


### Statistical Summary of Numerical Variables


In [ ]:
num_cols = raw_df.select_dtypes(include=[np.number]).columns.tolist()
raw_df[num_cols].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']]


### Anomaly & Quality Audit: Daily Activity Hours
A student's 24-hour day must naturally accommodate sleep, study, screen time, and exercise. We calculate total daily accounted hours:
$$\text{Total Hours} = \text{Avg\_Daily\_Usage\_Hours} + \text{Study\_Hours} + \text{Physical\_Activity\_Hours} + \text{Sleep\_Hours\_Per\_Night}$$


In [ ]:
raw_df['Total_Accounted_Hours'] = (
    raw_df['Avg_Daily_Usage_Hours'] + 
    raw_df['Study_Hours'] + 
    raw_df['Physical_Activity_Hours'] + 
    raw_df['Sleep_Hours_Per_Night']
)

print("Total Accounted Hours Summary:")
print(raw_df['Total_Accounted_Hours'].describe())
print(f"Any records exceeding 24 hours? {(raw_df['Total_Accounted_Hours'] > 24.0).sum()}")
print(f"Any negative physical activity hours? {(raw_df['Physical_Activity_Hours'] < 0.0).sum()}")


### Categorical Variables Distribution


In [ ]:
cat_cols = raw_df.select_dtypes(include=['object']).columns.tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

categorical_plots = ['Gender', 'Academic_Level', 'Purpose_Of_Use', 'Stress_Level']
for i, col in enumerate(categorical_plots):
    sns.countplot(data=raw_df, x=col, ax=axes[i], palette='Set2')
    axes[i].set_title(f"Distribution of {col}", fontweight='bold')
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Count")

plt.tight_layout()
plt.show()


### Target Variable Distribution (Mental Health Score)

We analyze the distribution of `Mental_Health_Score`. We define the 25th percentile threshold ($5.10$), which establishes the baseline cutoff for identifying students who may benefit from structured daily routine adjustments.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(raw_df['Mental_Health_Score'], kde=True, color='#1f77b4', bins=25, ax=ax1)
p25 = np.percentile(raw_df['Mental_Health_Score'], 25)
p50 = np.percentile(raw_df['Mental_Health_Score'], 50)
p75 = np.percentile(raw_df['Mental_Health_Score'], 75)

ax1.axvline(p25, color='orange', linestyle='--', label=f'25th Percentile ({p25:.2f})')
ax1.axvline(p50, color='red', linestyle='-', label=f'Median ({p50:.2f})')
ax1.axvline(p75, color='green', linestyle='--', label=f'75th Percentile ({p75:.2f})')
ax1.set_title("Mental Health Score Distribution", fontweight='bold')
ax1.legend()

sns.boxplot(y=raw_df['Mental_Health_Score'], color='#aec7e8', ax=ax2)
ax2.set_title("Mental Health Score Boxplot", fontweight='bold')
plt.show()


### Correlation Analysis


In [ ]:
plt.figure(figsize=(9, 7))
corr_matrix = raw_df[num_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, linewidths=0.5)
plt.title("Pearson Correlation Heatmap of Continuous Features", fontweight='bold', pad=15)
plt.xticks(rotation=30, ha='right')
plt.show()


## 6. Preprocessing & Prevention of Data Leakage

### 1. Data Sanitization:
- **Deduplication:** Remove exact duplicate rows (2 duplicate instances found).
- **Physical Activity Non-Negativity:** Physical exercise hours cannot logically be negative; 10 small negative values ($-0.1$ to $-0.4$) are clipped to $0.0$.

### 2. Preventing Target Leakage:
- `Stress_Level` represents a student's concurrent psychological strain—an outcome/symptom variable rather than an antecedent behavioral habit.
- Training models with `Stress_Level` would create severe target leakage. Therefore, **`Stress_Level` is strictly excluded from our primary feature set**, and analyzed separately in an ablation study.

### 3. Pipeline Separation:
- Preprocessing transformations (`StandardScaler` and `OneHotEncoder`) are fit strictly inside cross-validation folds using scikit-learn `Pipeline` and `ColumnTransformer` to avoid data leakage.


In [ ]:
# Clean the dataset
df_clean = raw_df.drop_duplicates().reset_index(drop=True)
df_clean['Physical_Activity_Hours'] = df_clean['Physical_Activity_Hours'].clip(lower=0.0)

print(f"Cleaned dataset shape: {df_clean.shape}")

# Define feature groups
NUMERICAL_FEATURES = [
    'Age', 'Avg_Daily_Usage_Hours', 'Daily_Unlocks',
    'Study_Hours', 'Physical_Activity_Hours', 'Sleep_Hours_Per_Night'
]
CATEGORICAL_FEATURES = [
    'Gender', 'Country', 'Academic_Level', 'Most_Used_Platform', 'Purpose_Of_Use'
]
TARGET_COLUMN = 'Mental_Health_Score'

X = df_clean[NUMERICAL_FEATURES + CATEGORICAL_FEATURES]
y = df_clean[TARGET_COLUMN]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Held-out test set: {X_test.shape[0]} samples")


## 7. Model Implementation & Benchmarking

We evaluate 7 models:
1. **Baseline 1:** Dummy Regressor (predicts mean of training set)
2. **Linear Regression (OLS):** Standard linear relationship
3. **Ridge Regression:** L2 regularized linear model
4. **Lasso Regression:** L1 regularized linear model
5. **K-Nearest Neighbors (KNN):** Distance-weighted local regression
6. **Random Forest Regressor:** Bagged ensemble of decision trees
7. **Gradient Boosting Regressor:** Sequential gradient boosted trees


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Construct preprocessors
preprocessor_scaled = ColumnTransformer([
    ('num', StandardScaler(), NUMERICAL_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES)
])

preprocessor_unscaled = ColumnTransformer([
    ('num', 'passthrough', NUMERICAL_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES)
])

# Candidate configurations
models = {
    'Baseline: Dummy (Mean)': (DummyRegressor(strategy='mean'), {}, preprocessor_unscaled),
    'Linear Regression': (LinearRegression(), {}, preprocessor_scaled),
    'Ridge Regression': (Ridge(random_state=42), {'regressor__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}, preprocessor_scaled),
    'Lasso Regression': (Lasso(random_state=42, max_iter=2000), {'regressor__alpha': [0.001, 0.01, 0.1]}, preprocessor_scaled),
    'K-Nearest Neighbors': (KNeighborsRegressor(), {'regressor__n_neighbors': [3, 5, 7, 9], 'regressor__weights': ['uniform', 'distance']}, preprocessor_scaled),
    'Random Forest': (RandomForestRegressor(random_state=42, n_jobs=-1), {'regressor__n_estimators': [100, 200], 'regressor__max_depth': [10, 15, None]}, preprocessor_unscaled),
    'Gradient Boosting': (GradientBoostingRegressor(random_state=42), {'regressor__n_estimators': [100, 200], 'regressor__learning_rate': [0.05, 0.1], 'regressor__max_depth': [3, 5]}, preprocessor_unscaled)
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
benchmark_results = []
fitted_estimators = {}

for name, (reg, params, prep) in models.items():
    pipe = Pipeline([('preprocessor', prep), ('regressor', reg)])
    if params:
        search = GridSearchCV(pipe, params, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1)
        search.fit(X_train, y_train)
        best_est = search.best_estimator_
        cv_rmse = -search.best_score_
        best_p = search.best_params_
    else:
        pipe.fit(X_train, y_train)
        best_est = pipe
        from sklearn.model_selection import cross_val_score
        cv_scores = cross_val_score(best_est, X_train, y_train, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=-1)
        cv_rmse = -np.mean(cv_scores)
        best_p = "Default"
        
    y_pred = best_est.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_mae = mean_absolute_error(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)
    
    fitted_estimators[name] = best_est
    benchmark_results.append({
        'Model': name,
        'CV_RMSE': round(cv_rmse, 4),
        'Test_RMSE': round(test_rmse, 4),
        'Test_MAE': round(test_mae, 4),
        'Test_R2': round(test_r2, 4),
        'Best_Hyperparameters': str(best_p)
    })

comparison_df = pd.DataFrame(benchmark_results).sort_values('Test_RMSE')
comparison_df


## 8. Visualizations & Residual Diagnostics


In [ ]:
best_model_name = comparison_df.iloc[0]['Model']
best_pipe = fitted_estimators[best_model_name]
y_test_pred = best_pipe.predict(X_test)
residuals = y_test - y_test_pred

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Actual vs Predicted
ax1.scatter(y_test, y_test_pred, alpha=0.4, color='#1f77b4', edgecolors='k', s=30)
min_v = min(y_test.min(), y_test_pred.min()) - 0.2
max_v = max(y_test.max(), y_test_pred.max()) + 0.2
ax1.plot([min_v, max_v], [min_v, max_v], 'r--', lw=2, label='1:1 Line')
ax1.set_title(f"Actual vs. Predicted ({best_model_name})", fontweight='bold')
ax1.set_xlabel("Actual Mental Health Score")
ax1.set_ylabel("Predicted Mental Health Score")
ax1.legend()

# Residual Plot
ax2.scatter(y_test_pred, residuals, alpha=0.4, color='#ff7f0e', edgecolors='k', s=30)
ax2.axhline(0, color='r', linestyle='--', lw=2)
ax2.set_title("Residuals vs. Fitted Values", fontweight='bold')
ax2.set_xlabel("Predicted Mental Health Score")
ax2.set_ylabel("Residual (Actual - Predicted)")

plt.tight_layout()
plt.show()


## 9. Country Generalization (GroupKFold) & Target Leakage Ablation

### 1. Generalization across Countries:
Does the model generalize to students from countries unseen during training? We use `GroupKFold(n_splits=5)` grouping on `Country`.


In [ ]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)
country_rmses, country_r2s = [], []

for train_idx, val_idx in gkf.split(X, y, groups=df_clean['Country']):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Train Ridge model as transparent benchmark
    pipe_c = Pipeline([('pre', preprocessor_scaled), ('reg', Ridge(alpha=1.0))])
    pipe_c.fit(X_tr, y_tr)
    preds = pipe_c.predict(X_val)
    country_rmses.append(np.sqrt(mean_squared_error(y_val, preds)))
    country_r2s.append(r2_score(y_val, preds))

print("=== Country Generalization (Leave-Countries-Out 5-Fold) ===")
print(f"Mean GroupKFold RMSE: {np.mean(country_rmses):.4f} +/- {np.std(country_rmses):.4f}")
print(f"Mean GroupKFold R2:   {np.mean(country_r2s):.4f} +/- {np.std(country_r2s):.4f}")


### 2. Target Leakage Ablation Study (With vs. Without Stress Level)


In [ ]:
# Compare Leakage-Free vs Leaked Model
preprocessor_leaked = ColumnTransformer([
    ('num', StandardScaler(), NUMERICAL_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES + ['Stress_Level'])
])

leaked_pipe = Pipeline([('pre', preprocessor_leaked), ('reg', Ridge(alpha=1.0))])
leaked_pipe.fit(df_clean.loc[X_train.index, NUMERICAL_FEATURES + CATEGORICAL_FEATURES + ['Stress_Level']], y_train)
leaked_preds = leaked_pipe.predict(df_clean.loc[X_test.index, NUMERICAL_FEATURES + CATEGORICAL_FEATURES + ['Stress_Level']])

print("Ablation Study Summary:")
print(f"Main Model (Leakage-Free) Test RMSE: {comparison_df.iloc[0]['Test_RMSE']:.4f} (R2: {comparison_df.iloc[0]['Test_R2']:.4f})")
print(f"Leaked Model (with Stress_Level) Test RMSE: {np.sqrt(mean_squared_error(y_test, leaked_preds)):.4f} (R2: {r2_score(y_test, leaked_preds):.4f})")


## 10. Conclusion, Limitations, and Future Scope

### Key Findings:
1. **Model Superiority:** The non-learning baseline ($R^2 = -0.0006$, RMSE $= 1.3365$) was vastly outperformed by all trained algorithms. K-Nearest Neighbors ($R^2 = 0.9245$, RMSE $= 0.3671$) and Random Forest ($R^2 = 0.9171$, RMSE $= 0.3847$) achieved the strongest predictive performance on tabular lifestyle patterns.
2. **Key Predictive Drivers:** Sleep duration ($r = +0.77$) and daily study routines ($r = +0.75$) emerged as the dominant positive predictors of student mental health, while daily usage hours ($r = -0.82$) and unlocks ($r = -0.79$) exhibited strong negative associations.
3. **Target Leakage:** Excluding `Stress_Level` preserved the integrity of the predictive task, ensuring all inputs are observable antecedents rather than outcome proxies.

### Limitations:
- Cross-sectional, self-reported data limits causal inference.
- Data patterns suggest clean synthetic regularities; validation on prospective clinical cohorts is recommended.

### Future Scope:
- Integration of continuous passive wearable sensor data (e.g., actigraphy for objective sleep and activity tracking).
- Longitudinal tracking across academic semesters to model temporal dynamics and burnout transitions.
